# **SQL Data Cleaning, Transformation and Analysis: Raw Data of Open-Pit Copper Mine**

As there are no publicly available raw datasets for open-pit copper mine operations, synthetic data representing an open-pit copper mine was generated using generative artificial intelligence.

Generative artificial intelligence was used solely for the creation of the synthetic dataset. All SQL data extraction, cleaning, transformation, and analysis presented in this project are entirely my own work. 

The relational data model for the dataset is shown in the image below and consists of five tables, representing the following information:

1. **`01_energy_sources_raw`** – Energy sources used across the mine's operations <br><br>
   
2. **`02_operation_hierarchy_raw`** – The operations and sub-operations within the mine <br><br>
  
3. **`03_daily_production_raw`** – Daily mine production data <br><br>
   
4. **`04_daily_energy_consumption_raw`** – Daily energy consumption across the mine's operations <br><br>
  
5. **`05_monthly_energy_costs_raw`** – Monthly energy costs associated with the various energy sources used across different mine operations <br><br>
   
![Relational Database](Relational%20Database.png)

The raw datasets contained inconsistent text formatting, mixed date formats, non-numeric values, missing values, different energy units, and values outside expected ranges. SQL was used to clean and transform the datasets before combining them for the final energy, cost and emissions analysis.

## **Table 1 (01_energy_sources_raw) - Data Cleaning & Transformation:**

A new table, **`energy_sources_clean`**, was created from the **`01_energy_sources_raw`** table using the `CREATE TABLE` statement. During this process, several data cleaning and transformation steps were applied:

* The `TRIM()` function was used to remove leading and trailing spaces, while the `UPPER()` function was applied to standardise the formatting of energy source names <br><br>

* Energy units were standardised using a `CASE` statement to ensure consistency across the dataset. For example, values such as 'mwh' were converted to 'MWh', while 'litres' was standardised to 'L' <br><br>

* The **`emissions_factor_tco2e_per_unit`** column represents the emissions factor for each energy source, expressed as tonnes of CO₂e per unit of energy consumed. <br><br>As the original emissions factors were reported using different units depending on the energy source, i.e. MWh, GJ, and L, a new column named **`emissions_factor_tco2e_per_MJ`** was created in order to standardise the emissions factor values <br><br>

* A new column, **`MJ_per_unit`**, was also created to store the conversion factor between each energy source's original unit and megajoules (MJ).<br><br>  The conversion factors used were: <br><br> 1 MWh = 3,6000 MJ <br><br> 1L = 40 MJ (Diesel) <br><br> 1 GJ = 1000 MJ <br><br> This conversion factor was then used to calculate the standardised values in the **`emissions_factor_tco2e_per_MJ`** column. <br><br> 

In [3]:
--TABLE 1 - ENERGY SOURCES
CREATE TABLE energy_sources_clean AS 
SELECT * 
FROM '01_energy_sources_raw.csv';


--sdlfkjsadf
UPDATE energy_sources_clean
	SET energy_source = UPPER(TRIM(energy_source)),
	
	standard_unit = CASE WHEN standard_unit ILIKE '%mwh%' THEN 'MWh'
	                     WHEN standard_unit ILIKE '%litres%' THEN 'L'
	                	 ELSE standard_unit END;

ALTER TABLE energy_sources_clean
	ADD COLUMN MJ_per_unit NUMERIC;

UPDATE energy_sources_clean
   SET MJ_per_unit = CASE WHEN standard_unit = 'MWh' THEN 3
	                      WHEN standard_unit = 'L'   THEN 40
	                      WHEN standard_unit = 'GJ'  THEN 1000
	                      ELSE MJ_per_unit END;

ALTER TABLE energy_sources_clean
	ADD COLUMN emissions_factor_tco2e_per_MJ DOUBLE;

UPDATE energy_sources_clean
   SET emissions_factor_tco2e_per_MJ = emissions_factor_tco2e_per_unit/MJ_per_unit;

SELECT *
FROM energy_sources_clean;

,energy_source_id,energy_source,standard_unit,emissions_factor_tco2e_per_unit,MJ_per_unit,emissions_factor_tco2e_per_MJ
0,16706266,GRID ELECTRICITY,MWh,0.69000,3600.0,0.000192
1,28450097,DIESEL,L,0.00268,40.0,0.000067
2,53385195,NATURAL GAS,GJ,0.05150,1000.0,0.000051
3,22757633,SOLAR PPA,MWh,0.03000,3600.0,0.000008


## **Table 2 (01_energy_sources_raw) - Data Cleaning & Transformation:**